In [ ]:
# Import libraries for data processing
import pandas as pd
import numpy as np

In [ ]:
# Read excel file
excel_file = pd.ExcelFile('/content/USECASE - Data Engineering.xlsx')

# Print sheet names
print(excel_file.sheet_names)

['product_details', 'retail_data1', 'retail_data2']


In [ ]:
# Load first retail dataset
retail1 = pd.read_excel(
    '/content/USECASE - Data Engineering.xlsx',
    sheet_name='retail_data1'
)

# Load second retail dataset
retail2 = pd.read_excel(
    '/content/USECASE - Data Engineering.xlsx',
    sheet_name='retail_data2'
)

# Load product details table
product = pd.read_excel(
    '/content/USECASE - Data Engineering.xlsx',
    sheet_name='product_details'
)

# Show number of rows and columns
print("Retail1 Shape:", retail1.shape)
print("Retail2 Shape:", retail2.shape)
print("Product Shape:", product.shape)

Retail1 Shape: (4243, 16)
Retail2 Shape: (4251, 16)
Product Shape: (10, 4)


In [ ]:
# Show column names
print(retail1.columns)

Index(['transaction_id', 'customer_id', 'customer_name', 'product_id', 'price',
       'product_name', 'category', 'purchase_location', 'city',
       'transaction_date', 'quantity', 'payment_method', 'discount', 'email',
       'phone', 'payment_status'],
      dtype='object')


In [ ]:
# Combine both retail datasets into one
final_data = pd.concat([retail1, retail2])

# Remove duplicate rows
final_data = final_data.drop_duplicates()

# Fill missing price with average price
final_data['price'] = final_data['price'].fillna(
    final_data['price'].mean()
)

# Fill missing quantity with 1
final_data['quantity'] = final_data['quantity'].fillna(1)

# Convert transaction date into proper format
final_data['transaction_date'] = pd.to_datetime(
    final_data['transaction_date'],
    errors='coerce'
)

# Remove invalid quantity (less than or equal to 0)
final_data = final_data[final_data['quantity'] > 0]

# Standardize product names (make lowercase)
final_data['product_name'] = (
    final_data['product_name']
    .astype(str)
    .str.lower()
    .str.strip()
)

# Mask customer email
final_data['email'] = (
    final_data['email']
    .astype(str)
    .str[:3] + "****"
)

# Mask phone number
final_data['phone'] = (
    "XXXXXX" +
    final_data['phone']
    .astype(str)
    .str[-4:]
)

# Create revenue column
final_data['revenue'] = (
    final_data['price'] *
    final_data['quantity']
)

# Show first 5 rows
final_data.head()

,transaction_id,customer_id,customer_name,product_id,price,product_name,category,purchase_location,city,transaction_date,quantity,payment_method,discount,email,phone,payment_status,revenue
0,1,642,Troy Mitchell,108,15000.0,mixer grinder,Home Appliances,offline,Bangalore,2025-12-06,3,UPI,0.10,Tro****,XXXXXX6968,successful,45000.0
1,2,881,Sarah Guerrero,102,70000.0,phone,ELEC,online,Bangalore,2025-07-21,5,Card,0.35,Sar****,XXXXXX8911,successful,350000.0
2,3,505,Samantha Hull,109,65000.0,refrigerator,Home Appliances,offline,Chennai,2025-07-11,2,UPI,0.05,Sam****,XXXXXX1565,successful,130000.0
3,4,794,Gerald Cooper,106,80000.0,sofa,FURN,offline,Chennai,2025-10-06,2,UPI,0.05,Ger****,XXXXXX4201,successful,160000.0
4,5,864,Cameron Black,108,15000.0,mixer grinder,home appliances,offline,Chennai,2025-12-12,1,Cash,0.05,Cam****,XXXXXX9477,successful,15000.0


In [ ]:
       #. KPI Calculation


# Total revenue
total_revenue = final_data['revenue'].sum()

print("Total Revenue:", total_revenue)

# Revenue by category
revenue_category = (
    final_data
    .groupby('category')['revenue']
    .sum()
)

print("\nRevenue By Category")
print(revenue_category)

# Revenue by city
revenue_city = (
    final_data
    .groupby('city')['revenue']
    .sum()
)

print("\nRevenue By City")
print(revenue_city)

Total Revenue: 1605314642.0689657

Revenue By Category
category
CLOTH              1.741411e+07
Clothing           1.734283e+07
ELEC               3.217646e+08
Electronics        2.913849e+08
FURN               1.015465e+08
Furniture          1.142263e+08
HOME               1.109540e+08
Home Appliances    1.083353e+08
clothing           1.531444e+07
electronics        2.839034e+08
furniture          1.132051e+08
home appliances    1.099232e+08
Name: revenue, dtype: float64

Revenue By City
city
Bangalore    3.017355e+08
Chennai      3.291912e+08
Delhi        3.335590e+08
Hyderabad    3.197595e+08
Mumbai       3.210694e+08
Name: revenue, dtype: float64


In [ ]:
# Save cleaned dataset as CSV
final_data.to_csv(
    "cleaned_retail_data.csv",
    index=False
)

print("File Saved Successfully")

File Saved Successfully
